In [13]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import seaborn as sns
from datetime import datetime
import warnings
import kagglehub

## 1. Project Overview

Hypothesis Summary: Weather and Tourist Arrivals in Singapore

**Research question:** To what extent do weather conditions in Singapore explain changes in tourist arrivals, and how can the Singapore Tourism Board (STB) use this analysis to increase tourism?

### Hypotheses

| | Statement |
|---|---|
| **H₀ (null)** | Weather conditions (rainfall, rainy days, temperature, sunshine, humidity) have **no significant effect** on monthly tourist arrivals to Singapore. |
| **H₁ (working hypothesis)** | Weather conditions, such as the monsoon seasons, **significantly influence** monthly tourist arrival volumes. |

**Type of Analysis**: This project will utilize Time-Series Regression Analysis. While Singapore possesses a relatively static tropical climate year-round, granular weather data (e.g., monthly rainfall, humidity, and temperature variations) exhibits certain level of variance to impact travel decisions. By analyzing historical weather and arrival data, we aim to build a model that predicts arrival peaks and troughs.

**Targeted Quality Metric**: The performance of our predictive model will be evaluated using Mean Absolute Error (MAE) to understand the average deviation in our predicted visitor counts, alongside R-squared ($R^2$) to determine the proportion of variance in tourist arrivals that can be explained by our weather variables.

### Data Used
- SingStat monthly weather data (rainfall, rainy days, max temperature, sunshine hours, humidity) 
- SingStat monthly tourist arrivals, 2008–2025
- The Kaggle Global Weather Repository, used as secondary source to validate the SingStat monthly weather data.



## 2.Data Source Identification and EDA


*   **Singapore Weather Data**
*   [Global Weather Repository (DS1)](https://www.kaggle.com/datasets/nelgiriyewithana/global-weather-repository/data/discussion)
*   [Singapore Air Temperature, Sunshine, Relative Humidity and Rainfall in Singapore (DS2)](https://tablebuilder.singstat.gov.sg/table/TS/M890081)
*   **Singapore Tourism Data**
*   [Singapore Tourist Arrival (DS3)](https://tablebuilder.singstat.gov.sg/table/TS/M5550001)

In [14]:
import kagglehub
import glob
import os
import pandas as pd
import numpy as np

# Find the root of the GitHub repository (the folder containing .git),
# so the data always lands inside the repo no matter where Jupyter was started
def find_repo_root(start=os.getcwd()):
    current = os.path.abspath(start)
    while True:
        if os.path.isdir(os.path.join(current, ".git")):
            return current
        parent = os.path.dirname(current)
        if parent == current:          # reached the drive root without finding .git
            return os.path.abspath(start)
        current = parent

REPO_ROOT = find_repo_root()
DATA_DIR = os.path.join(REPO_ROOT, "data")
os.makedirs(DATA_DIR, exist_ok=True)

# Pin a specific dataset version so everyone gets the same data
# (the dataset is updated daily; the latest version 1106 on Kaggle is an empty archive)
KAGGLE_DATASET = "nelgiriyewithana/global-weather-repository/versions/1105"

# Only download from Kaggle if the CSV is not already in the repository's data folder
csv_files = glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True)
if not csv_files:
    print("Downloading dataset from Kaggle into:", DATA_DIR)
    kagglehub.dataset_download(
        KAGGLE_DATASET,
        output_dir=DATA_DIR,       # save straight into the repo instead of the kagglehub cache
        force_download=True,
    )
    csv_files = glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True)

if not csv_files:
    raise FileNotFoundError(f"No CSV found in {DATA_DIR} after downloading. Check your internet connection / Kaggle login.")

print("Path to dataset files:", DATA_DIR)

# Load the CSV from the repository's data folder
file_path = csv_files[0]
print("Loading:", file_path)
df = pd.read_csv(file_path)
print(df.shape)

Path to dataset files: c:\Users\wenju\Data-Processing\data
Loading: c:\Users\wenju\Data-Processing\data\GlobalWeatherRepository.csv
(170733, 41)


##### 2.3.1.1 Data Profiling for Source 1

In [15]:
# Data is loaded from Kaggle in the cell above (df)
country_col = df.columns[0]

#We only want data for Singapore
singapore_data = df[df[country_col].astype(str).str.strip() == 'Singapore']
singapore_data.to_csv('singapore_only.csv', index=False)

print("Data Profiling for Source 1: Global Weather Repository")
print("="*60)

df = singapore_data  # your existing Source 1 dataframe

print("\n1. STRUCTURE INSPECTION")
print("-" * 60)
print(f"Source 1 has {df.shape[0]} rows and {df.shape[1]} columns")

print("\n2. DATA TYPES & MISSING VALUES")
print("-" * 60)
print("\nColumn Data Types:")
print(df.dtypes)
print("\nMissing Values:")
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_summary = pd.DataFrame({
    'Missing Count': missing,
    'Percentage': missing_pct
})
print(missing_summary[missing_summary['Missing Count'] > 0])

print("\n3. STATISTICAL PROFILING (Numeric Columns)")
print("-" * 60)
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print(f"\nNumeric columns found: {len(numeric_cols)}")
print(numeric_cols[:10])
if len(numeric_cols) > 0:
    print("\nSummary Statistics:")
    print(df[numeric_cols].describe())

print("\n4. CATEGORICAL COLUMNS ANALYSIS")
print("-" * 60)
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
print(f"\nCategorical columns found: {len(categorical_cols)}")
for col in categorical_cols[:5]:
    print(f"\n'{col}' - Unique values: {df[col].nunique()}")
    if df[col].nunique() <= 20:
        print(df[col].value_counts())
    else:
        print(f"Top 10 values:")
        print(df[col].value_counts().head(10))

print("\n5. DATE/DATETIME ANALYSIS")
print("-" * 60)
date_cols = df.select_dtypes(include=['datetime64']).columns.tolist()
for col in df.columns:
    if 'date' in col.lower() or 'time' in col.lower():
        if col not in date_cols:
            try:
                df[col + '_converted'] = pd.to_datetime(df[col], errors='coerce')
                if df[col + '_converted'].notna().sum() > len(df) * 0.5:
                    date_cols.append(col)
                    print(f"Found potential date column: '{col}'")
            except:
                pass
if len(date_cols) > 0:
    for col in date_cols:
        if col in df.columns:
            print(f"\n'{col}':")
            try:
                date_range = df[col].max() - df[col].min()
                print(f"  Date range: {df[col].min()} to {df[col].max()}")
                if hasattr(date_range, 'days'):
                    print(f"  Total days: {date_range.days}")
            except:
                print(f"  Range: {df[col].min()} to {df[col].max()}")
else:
    print("No date columns detected.")

print("\n6. DATA QUALITY CHECKS")
print("-" * 60)
duplicates = df.duplicated().sum()
print(f"Duplicate rows: {duplicates}")
if len(numeric_cols) > 0:
    print("\nZero values in numeric columns:")
    for col in numeric_cols[:10]:
        zero_count = (df[col] == 0).sum()
        if zero_count > 0:
            print(f"  {col}: {zero_count} zeros ({zero_count/len(df)*100:.2f}%)")

print("\n7. CORRELATION ANALYSIS (Top Numeric Columns)")
print("-" * 60)
if len(numeric_cols) >= 2:
    corr_cols = numeric_cols[:min(10, len(numeric_cols))]
    corr_matrix = df[corr_cols].corr()
    strong_corr = []
    for i in range(len(corr_matrix.columns)):
        for j in range(i+1, len(corr_matrix.columns)):
            corr_val = corr_matrix.iloc[i, j]
            if abs(corr_val) > 0.7:
                strong_corr.append((corr_matrix.columns[i], corr_matrix.columns[j], corr_val))
    if strong_corr:
        print("\nStrong correlations found (|r| > 0.7):")
        for col1, col2, val in strong_corr[:10]:
            print(f"  {col1} <-> {col2}: {val:.3f}")
    else:
        print("No strong correlations found (|r| > 0.7)")
else:
    print("Insufficient numeric columns for correlation analysis.")

print("\n8. UNIQUE IDENTIFIERS CHECK")
print("-" * 60)
for col in df.columns[:10]:
    unique_ratio = df[col].nunique() / len(df)
    if unique_ratio == 1.0:
        print(f"'{col}' - Potential unique identifier (100% unique)")
    elif unique_ratio > 0.95:
        print(f"'{col}' - High uniqueness ({unique_ratio*100:.1f}%)")

print("\n9. DATA QUALITY DISCUSSION (Balanced Evaluation)")
print("-" * 60)

overall_missing_pct = (df.isnull().sum().sum() / (df.shape[0] * df.shape[1])) * 100

print("\nCompleteness:")
if overall_missing_pct < 5:
    print(f"  (+) The dataset is highly complete with only {overall_missing_pct:.2f}% missing values, reducing the need for extensive data cleaning.")
else:
    print(f"  (-) The dataset has {overall_missing_pct:.2f}% missing values, which may require imputation or removal of incomplete records during preprocessing.")

print("\nReliability:")
print(f"  (-) Data is downloaded from \"Kaggle.\" and hence might not be as reliable as those other data from governmental sources. Cross referencing is required.")

print("\nTimeliness:")
if len(date_cols) > 0:
    try:
        # Convert Unix timestamp to datetime
        min_ts = df[date_cols[0]].min()
        max_ts = df[date_cols[0]].max()

        min_date = pd.to_datetime(min_ts, unit='s')
        max_date = pd.to_datetime(max_ts, unit='s')

        print(f"  (+) The dataset is updated daily from {min_date} to {max_date}, allowing for temporal analysis.")
    except:
        print("  (-) Date columns exist but could not be converted properly.")
else:
    print("  (-) No date information available.")


Data Profiling for Source 1: Global Weather Repository

1. STRUCTURE INSPECTION
------------------------------------------------------------
Source 1 has 877 rows and 41 columns

2. DATA TYPES & MISSING VALUES
------------------------------------------------------------

Column Data Types:
country                             str
location_name                       str
latitude                        float64
longitude                       float64
timezone                            str
last_updated_epoch                int64
last_updated                        str
temperature_celsius             float64
temperature_fahrenheit          float64
condition_text                      str
wind_mph                        float64
wind_kph                        float64
wind_degree                       int64
wind_direction                      str
pressure_mb                     float64
pressure_in                     float64
precip_mm                       float64
precip_in                    

C:\Users\wenju\AppData\Local\Temp\ipykernel_63556\2113753674.py:41: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df.select_dtypes(include=['object']).columns.tolist()


##### 2.3.1.2 Data Structuring for Source 1

In [16]:
print("Data Structuring for Source 1: Global Weather Repository")
print("="*60)

# Filter to Singapore only
df_s1 = df[df['country'].str.lower() == 'singapore'].copy()
print(f"\nRows after Singapore filter: {len(df_s1)}")

# Parse datetime column
df_s1['last_updated'] = pd.to_datetime(df_s1['last_updated'], errors='coerce')
invalid_dates = df_s1['last_updated'].isna().sum()
print(f"Invalid datetime entries removed: {invalid_dates}")
df_s1 = df_s1[df_s1['last_updated'].notna()].copy()

# Extract month period for aggregation
df_s1['month'] = df_s1['last_updated'].dt.to_period('M')

# Keep only relevant columns for analysis
s1_keep = ['month', 'temperature_celsius', 'humidity', 'precip_mm']
df_s1 = df_s1[s1_keep].copy()

# Standardise column names
df_s1.rename(columns={
    'temperature_celsius' : 'temp_celsius',
    'humidity'            : 'humidity_pct',
    'precip_mm'           : 'rainfall_mm'
}, inplace=True)

print(f"\nColumns retained : {df_s1.columns.tolist()}")
print(f"Shape            : {df_s1.shape}")
print(f"Date range       : {df_s1['month'].min()} to {df_s1['month'].max()}")
print("\nFirst 5 rows:")
print(df_s1.head())

Data Structuring for Source 1: Global Weather Repository

Rows after Singapore filter: 877
Invalid datetime entries removed: 0

Columns retained : ['month', 'temp_celsius', 'humidity_pct', 'rainfall_mm']
Shape            : (877, 4)
Date range       : 2024-05 to 2026-10

First 5 rows:
       month  temp_celsius  humidity_pct  rainfall_mm
156  2024-05          26.0            89         1.75
351  2024-05          27.0            94         0.00
545  2024-05          28.0            89         0.52
737  2024-05          29.0            75         0.00
932  2024-05          29.0            79         0.35


##### 2.3.1.3 Data Enriching for Source 1

In [17]:
print("Data Enriching for Source 1: Global Weather Repository")
print("="*60)

# Aggregate daily readings to monthly, since Source 2 and 3 are both in monthly granaluarity
print("\nAggregating daily to monthly...")
df_s1_monthly = df_s1.groupby('month').agg(
    temp_celsius = ('temp_celsius', 'mean'),
    humidity_pct = ('humidity_pct', 'mean'),
    rainfall_mm  = ('rainfall_mm',  'sum')   # total monthly rainfall
).reset_index()

# Derive month number and year
df_s1_monthly['year']      = df_s1_monthly['month'].dt.year
df_s1_monthly['month_num'] = df_s1_monthly['month'].dt.month

# Add Singapore meteorological season
def get_season(m):
    if m in [12, 1, 2]:        return 'Northeast Monsoon'
    elif m in [3, 4]:           return 'Inter-monsoon 1'
    elif m in [5, 6, 7, 8, 9]: return 'Southwest Monsoon'
    else:                       return 'Inter-monsoon 2'

df_s1_monthly['season'] = df_s1_monthly['month_num'].apply(get_season)

# 3-month rolling averages for key variables
df_s1_monthly['temp_3m_avg']     = df_s1_monthly['temp_celsius'].rolling(window=3, min_periods=1).mean().round(2)
df_s1_monthly['humidity_3m_avg'] = df_s1_monthly['humidity_pct'].rolling(window=3, min_periods=1).mean().round(2)
df_s1_monthly['rainfall_3m_avg'] = df_s1_monthly['rainfall_mm'].rolling(window=3, min_periods=1).mean().round(2)

print(f"Shape after aggregation : {df_s1_monthly.shape}")
print(f"New columns added       : year, month_num, season, temp_3m_avg, humidity_3m_avg, rainfall_3m_avg")
print("\nFirst 5 rows:")
print(df_s1_monthly.head())

Data Enriching for Source 1: Global Weather Repository

Aggregating daily to monthly...
Shape after aggregation : (30, 10)
New columns added       : year, month_num, season, temp_3m_avg, humidity_3m_avg, rainfall_3m_avg

First 5 rows:
     month  temp_celsius  humidity_pct  rainfall_mm  year  month_num  \
0  2024-05     28.294118     83.529412         7.36  2024          5   
1  2024-06     28.360000     78.366667         6.65  2024          6   
2  2024-07     28.896552     74.551724         1.29  2024          7   
3  2024-08     28.809677     73.000000         0.52  2024          8   
4  2024-09     29.693333     68.433333        11.28  2024          9   

              season  temp_3m_avg  humidity_3m_avg  rainfall_3m_avg  
0  Southwest Monsoon        28.29            83.53             7.36  
1  Southwest Monsoon        28.33            80.95             7.00  
2  Southwest Monsoon        28.52            78.82             5.10  
3  Southwest Monsoon        28.69            75.31  

##### 2.3.1.4 Data Cleaning for Source 1

In [18]:
print("Data Cleaning for Source 1: Global Weather Repository")
print("="*60)

# Remove duplicate months
before = len(df_s1_monthly)
df_s1_monthly = df_s1_monthly.drop_duplicates(subset='month').sort_values('month').reset_index(drop=True)
print(f"\nDuplicate months removed : {before - len(df_s1_monthly)}")

# Validate realistic ranges for Singapore weather
print("\nApplying range validation...")
invalid_temp     = (~df_s1_monthly['temp_celsius'].between(20, 40)).sum()
invalid_humidity = (~df_s1_monthly['humidity_pct'].between(0, 100)).sum()
invalid_rainfall = (df_s1_monthly['rainfall_mm'] < 0).sum()

df_s1_monthly.loc[~df_s1_monthly['temp_celsius'].between(20, 40),  'temp_celsius'] = np.nan
df_s1_monthly.loc[~df_s1_monthly['humidity_pct'].between(0, 100),  'humidity_pct'] = np.nan
df_s1_monthly.loc[df_s1_monthly['rainfall_mm'] < 0,                'rainfall_mm']  = np.nan

print(f"  temp_celsius  : {invalid_temp} out-of-range values replaced with NaN")
print(f"  humidity_pct  : {invalid_humidity} out-of-range values replaced with NaN")
print(f"  rainfall_mm   : {invalid_rainfall} negative values replaced with NaN")

# Interpolate missing values (if there even is any)
print("\nInterpolating missing values...")
df_s1_monthly[['temp_celsius', 'humidity_pct', 'rainfall_mm']] = (
    df_s1_monthly[['temp_celsius', 'humidity_pct', 'rainfall_mm']]
    .interpolate(method='linear', limit_direction='both')
)

print(f"Remaining nulls : {df_s1_monthly.isnull().sum().sum()}")
print(f"Final shape     : {df_s1_monthly.shape}")
print("\nFirst 5 rows:")
print(df_s1_monthly.head())

Data Cleaning for Source 1: Global Weather Repository

Duplicate months removed : 0

Applying range validation...
  temp_celsius  : 0 out-of-range values replaced with NaN
  humidity_pct  : 0 out-of-range values replaced with NaN
  rainfall_mm   : 0 negative values replaced with NaN

Interpolating missing values...
Remaining nulls : 0
Final shape     : (30, 10)

First 5 rows:
     month  temp_celsius  humidity_pct  rainfall_mm  year  month_num  \
0  2024-05     28.294118     83.529412         7.36  2024          5   
1  2024-06     28.360000     78.366667         6.65  2024          6   
2  2024-07     28.896552     74.551724         1.29  2024          7   
3  2024-08     28.809677     73.000000         0.52  2024          8   
4  2024-09     29.693333     68.433333        11.28  2024          9   

              season  temp_3m_avg  humidity_3m_avg  rainfall_3m_avg  
0  Southwest Monsoon        28.29            83.53             7.36  
1  Southwest Monsoon        28.33            80.

##### 2.3.1.5 Potential areas to conduct further analysis

1. **Time Series Analysis:** Examine trends over the 622-day period
2. **Correlation Studies:** Investigate relationships between variables (e.g., temperature vs. humidity)
3. **Seasonal Decomposition:** Identify cyclical patterns
4. **Extreme Event Prediction:** Model conditions leading to heavy rain or high UV
5. **Air Quality Modeling:** Analyze factors contributing to PM2.5 spikes
6. **Comparison:** Compare with historical Singapore climate data






#### 2.3.2 Dataset 2 (DS2): Singapore Air Temperature, Sunshine, Relative Humidity, and Rainfall in Singapore

Descripton: This dataset is sourced from the National Environment Agency (NEA) via Singapore's Department of Statistics (SingStat Table Builder M890081), provides comprehensive monthly meteorological data for Singapore. The dataset includes 10 weather variables across temperature metrics (daily maximum/minimum means and absolute extremes), precipitation measures (total rainfall, highest daily rainfall, and number of rainy days), and environmental conditions (daily sunshine hours and relative humidity).

##### 2.3.2.1 Data Profiling for Source 2

In [19]:
print("Data Profiling for Source 2: Singapore Air Temperature")
print("="*60)

file_path2 = 'https://docs.google.com/spreadsheets/d/e/2PACX-1vT3x74IjOTTosLV72A8OaUjT_hSMZHHbjx55VazZJ9DD8nPnkjpOZhnxrTDHQlTjA/pub?output=csv'
df_raw = pd.read_csv(file_path2)
for col in df_raw.columns[1:]:
    df_raw[col] = pd.to_numeric(df_raw[col], errors='coerce')
df_s2 = df_raw.copy()

# Transpose the data as the months are now in the first row as the months are in the first row
df_temp = df_raw.set_index(df_raw.columns[0])
df_s2_ts = df_temp.T
df_s2_ts.index.name = 'month'
df_s2 = df_s2_ts.reset_index()
df_s2['month'] = pd.to_datetime(df_s2['month'], format='%Y-%m',errors='coerce')
df_s2 = df_s2.set_index('month')
df = df_s2

print("\n1. STRUCTURE INSPECTION")
print("-" * 60)
print(f"Source 2 has {df.shape[0]} rows and {df.shape[1]} columns")

print("\n2. DATA TYPES & MISSING VALUES")
print("-" * 60)
print("\nColumn Data Types:")
print(df.dtypes)
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_summary = pd.DataFrame({'Missing Count': missing, 'Percentage': missing_pct})
print("\nMissing Values:")
print(missing_summary[missing_summary['Missing Count'] > 0])
print(f"\nTotal data points   : {df.size}")
print(f"Missing data points : {df.isnull().sum().sum()}")
print(f"Data completeness   : {(1 - df.isnull().sum().sum()/df.size)*100:.2f}%")

print("\n3. STATISTICAL PROFILING (Numeric Columns)")
print("-" * 60)
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print(f"\nNumeric columns found: {len(numeric_cols)}")
print(numeric_cols[:10])
if len(numeric_cols) > 0:
    print("\nSummary Statistics:")
    print(df[numeric_cols].describe())

print("\n4. CATEGORICAL COLUMNS ANALYSIS")
print("-" * 60)
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
print(f"\nCategorical columns found: {len(categorical_cols)}")
for col in categorical_cols[:5]:
    print(f"\n'{col}' - Unique values: {df[col].nunique()}")
    if df[col].nunique() <= 20:
        print(df[col].value_counts())
    else:
        print("Top 10 values:")
        print(df[col].value_counts().head(10))

print("\n5. DATE/DATETIME ANALYSIS")
print("-" * 60)
print(f"Index name : '{df.index.name}'")
try:
    index_as_dates = pd.to_datetime(df.index, format='%Y %b', errors='coerce')
    valid_dates = index_as_dates.notna().sum()
    if valid_dates > len(df) * 0.5:
        print(f"  Index is a date/time series")
        print(f"  Date range  : {index_as_dates.min()} to {index_as_dates.max()}")
        date_range = index_as_dates.max() - index_as_dates.min()
        if hasattr(date_range, 'days'):
            print(f"  Total days  : {date_range.days}")
except:
    print(f"  Range: {df.index.min()} to {df.index.max()}")

print("\n6. DATA QUALITY CHECKS")
print("-" * 60)
print(f"Duplicate rows: {df.duplicated().sum()}")
if len(numeric_cols) > 0:
    print("\nZero values in numeric columns:")
    for col in numeric_cols[:10]:
        zero_count = (df[col] == 0).sum()
        if zero_count > 0:
            print(f"  {col}: {zero_count} zeros ({zero_count/len(df)*100:.2f}%)")

print("\n7. CORRELATION ANALYSIS (Top Numeric Columns)")
print("-" * 60)
if len(numeric_cols) >= 2:
    corr_cols = numeric_cols[:min(10, len(numeric_cols))]
    corr_matrix = df[corr_cols].corr()
    strong_corr = []
    for i in range(len(corr_matrix.columns)):
        for j in range(i+1, len(corr_matrix.columns)):
            corr_val = corr_matrix.iloc[i, j]
            if abs(corr_val) > 0.7:
                strong_corr.append((corr_matrix.columns[i], corr_matrix.columns[j], corr_val))
    if strong_corr:
        print("\nStrong correlations found (|r| > 0.7):")
        for col1, col2, val in strong_corr[:10]:
            print(f"  {col1} <-> {col2}: {val:.3f}")
    else:
        print("No strong correlations found (|r| > 0.7)")
    print("\nData Variability (CV per variable):")
    for col in numeric_cols[:10]:
        std  = df[col].std()
        mean = df[col].mean()
        cv   = (std / mean) * 100 if mean != 0 else 0
        print(f"  {col}: Mean={mean:.2f}, Std={std:.2f}, CV={cv:.1f}%")
else:
    print("Insufficient numeric columns for correlation analysis.")

print("\n8. UNIQUE IDENTIFIERS CHECK")
print("-" * 60)
for col in df.columns[:10]:
    unique_ratio = df[col].nunique() / len(df)
    if unique_ratio == 1.0:
        print(f"'{col}' - Potential unique identifier (100% unique)")
    elif unique_ratio > 0.95:
        print(f"'{col}' - High uniqueness ({unique_ratio*100:.1f}%)")

print("\n9. DATA QUALITY DISCUSSION (Source 2)")
print("-" * 60)

overall_missing_pct = (df.isnull().sum().sum() / df.size) * 100

print("\nCompleteness:")
if overall_missing_pct < 5:
    print(f"  (+) The dataset is highly complete with only {overall_missing_pct:.2f}% missing values, reducing the need for extensive data cleaning. These missing values come mainly from the \"Minimum relative humidity percent\".")
else:
    print(f"  (-) The dataset has {overall_missing_pct:.2f}% missing values, which may require imputation or removal of incomplete records.")


print("\nReliability:")
print(f"  (+) Data is downloaded from Singapore government backed agency and hence is much more reliable.")

# --- Consistency (Limitation) ---
print("\nConsistency:")
print("  (-) The dataset required transformation (transpose and type conversion), indicating that the original format was not analysis-r0eady and may introduce preprocessing complexity.")


Data Profiling for Source 2: Singapore Air Temperature

1. STRUCTURE INSPECTION
------------------------------------------------------------
Source 2 has 613 rows and 10 columns

2. DATA TYPES & MISSING VALUES
------------------------------------------------------------

Column Data Types:
Data Series
Air Temperature Means Daily Maximum (Degree Celsius)          float64
Air Temperature Means Daily Minimum (Degree Celsius)          float64
Air Temperature Absolute Extremes Maximum (Degree Celsius)    float64
Air Temperature Absolute Extremes Minimum (Degree Celsius)    float64
Total Rainfall (Millimetre)                                   float64
Highest Daily Rainfall Total (Millimetre)                     float64
Number Of Rainy Days (Number)                                 float64
Bright Sunshine Daily Mean (Hour)                             float64
Minimum Relative Humidity (Per Cent)                          float64
24 Hours Mean Relative Humidity (Per Cent)                    float

##### 2.3.2.2 Data Structuring for Source 2

In [20]:
print("Data Structuring for Source 2: Singapore Air Temperature")
print("="*60)

df_s2 = df_s2_ts.copy()

# Clean index first (There is one space after the month)
df_s2.index = df_s2.index.str.strip()
df_s2.index = pd.to_datetime(df_s2.index, format='%Y %b', errors='coerce').to_period('M')

invalid_idx = df_s2.index.isna().sum()
print(f"\nInvalid month index entries removed: {invalid_idx}")
df_s2 = df_s2[df_s2.index.notna()].copy()

# Bring month into a column and standardise column names
df_s2 = df_s2.reset_index()
df_s2.columns = [col.strip().lower().replace(' ', '_').replace('(', '').replace(')', '') for col in df_s2.columns]

print(f"Columns  : {df_s2.columns.tolist()}")
print(f"Shape    : {df_s2.shape}")
print(f"Date range: {df_s2['month'].min()} to {df_s2['month'].max()}")

print("\nFirst 5 rows:")
print(df_s2.head())

Data Structuring for Source 2: Singapore Air Temperature

Invalid month index entries removed: 0
Columns  : ['month', 'air_temperature_means_daily_maximum_degree_celsius', 'air_temperature_means_daily_minimum_degree_celsius', 'air_temperature_absolute_extremes_maximum_degree_celsius', 'air_temperature_absolute_extremes_minimum_degree_celsius', 'total_rainfall_millimetre', 'highest_daily_rainfall_total_millimetre', 'number_of_rainy_days_number', 'bright_sunshine_daily_mean_hour', 'minimum_relative_humidity_per_cent', '24_hours_mean_relative_humidity_per_cent']
Shape    : (613, 11)
Date range: 1975-01 to 2026-01

First 5 rows:
     month  air_temperature_means_daily_maximum_degree_celsius  \
0  2026-01                                               31.0    
1  2025-12                                               31.0    
2  2025-11                                               32.8    
3  2025-10                                               32.9    
4  2025-09                           

##### 2.3.2.3 Data Enriching for Source 2

In [21]:
print("Data Enriching for Source 2: Singapore Air Temperature")
print("="*60)

df_s2 = df_s2.sort_values('month').reset_index(drop=True)

# Extract year and month number
df_s2['year']      = df_s2['month'].dt.year
df_s2['month_num'] = df_s2['month'].dt.month

# Singapore meteorological season
def get_season(m):
    if m in [12, 1, 2]:        return 'Northeast Monsoon'
    elif m in [3, 4]:           return 'Inter-monsoon 1'
    elif m in [5, 6, 7, 8, 9]: return 'Southwest Monsoon'
    else:                       return 'Inter-monsoon 2'

df_s2['season'] = df_s2['month_num'].apply(get_season)

# 3-month rolling average for all numeric temperature columns
numeric_cols = df_s2.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c not in ['year', 'month_num']]
for col in numeric_cols:
    df_s2[f'{col}_3m_avg'] = df_s2[col].rolling(window=3, min_periods=1).mean().round(2)

print(f"New columns added : year, month_num, season, 3m rolling averages for {len(numeric_cols)} weather variable(s)")
print(f"Shape             : {df_s2.shape}")

print("\nFirst 5 rows:")
print(df_s2.head())

Data Enriching for Source 2: Singapore Air Temperature
New columns added : year, month_num, season, 3m rolling averages for 10 weather variable(s)
Shape             : (613, 24)

First 5 rows:
     month  air_temperature_means_daily_maximum_degree_celsius  \
0  1975-01                                               30.5    
1  1975-02                                               30.4    
2  1975-03                                               31.1    
3  1975-04                                               31.1    
4  1975-05                                               31.4    

   air_temperature_means_daily_minimum_degree_celsius  \
0                                               22.8    
1                                               23.0    
2                                               23.1    
3                                               23.8    
4                                               24.0    

   air_temperature_absolute_extremes_maximum_degree_celsius  \
0    

There are NaN values for the "minimum_relative_humidity_per_cent" and "minimum_relative_humidity_per_cent_3m_avg", which are expected as this issue has already been highlighted beforehand and likely due to historical limitations in data collection during the earlier years of the dataset.

There are missing values for the early 1970s and will not be an issue for us given that the focus of this analysis is on more recent climate trends.


##### 2.3.2.4 Data Cleaning for Source 2

In [22]:
print("Data Cleaning for Source 2: Singapore Air Temperature")
print("="*60)

# Initial NaN count
print("\nInitial missing values (NaN):")
print(df_s2.isnull().sum())
print(f"Total NaNs (before): {df_s2.isnull().sum().sum()}")

# Remove duplicate months
before = len(df_s2)
df_s2 = df_s2.drop_duplicates(subset='month').sort_values('month').reset_index(drop=True)
print(f"\nDuplicate months removed : {before - len(df_s2)}")

temp_cols     = [c for c in df_s2.columns if 'temperature' in c]
rainfall_cols = [c for c in df_s2.columns if 'rainfall' in c]
rain_days_cols= [c for c in df_s2.columns if 'rainy_days' in c or 'rainy days' in c.replace('_', ' ')]
sunshine_cols = [c for c in df_s2.columns if 'sunshine' in c]
humidity_cols = [c for c in df_s2.columns if 'humidity' in c]

print(f"\nColumn groups detected:")
print(f"  Temperature  : {temp_cols}")
print(f"  Rainfall     : {rainfall_cols}")
print(f"  Rainy days   : {rain_days_cols}")
print(f"  Sunshine     : {sunshine_cols}")
print(f"  Humidity     : {humidity_cols}")

# Apply validation rules
print("\nApplying validation rules...")

def report_invalid(col, condition, label):
    invalid = (condition & df_s2[col].notna()).sum()
    df_s2.loc[condition, col] = np.nan
    if invalid > 0:
        print(f"  {col} ({label}): {invalid} values replaced with NaN")

for col in temp_cols:
    report_invalid(col, ~df_s2[col].between(20, 40), "Temp 20–40°C")

for col in rainfall_cols:
    report_invalid(col, df_s2[col] < 0, "Rainfall >=0")

for col in rain_days_cols:
    report_invalid(col, ~df_s2[col].between(0, 31), "Rainy Days 0–31")

for col in sunshine_cols:
    report_invalid(col, ~df_s2[col].between(0, 12), "Sunshine 0–12h")

for col in humidity_cols:
    report_invalid(col, ~df_s2[col].between(0, 100), "Humidity 0–100%")

print("\nMissing values after validation:")
print(df_s2.isnull().sum())
print(f"Total NaNs (after validation): {df_s2.isnull().sum().sum()}")

all_weather_cols = temp_cols + rainfall_cols + rain_days_cols + sunshine_cols
mean_hum_cols    = [c for c in humidity_cols if 'minimum' not in c]
min_hum_cols     = [c for c in humidity_cols if 'minimum' in c]

print("\nInterpolating missing values (temperature + mean weather cols, limit=2)...")
df_s2[temp_cols] = df_s2[temp_cols].interpolate(method='linear', limit_direction='both')
df_s2[all_weather_cols + mean_hum_cols] = (
    df_s2[all_weather_cols + mean_hum_cols]
    .interpolate(method='linear', limit=2, limit_direction='both')
)
print(f"  Note: minimum relative humidity columns ({min_hum_cols}) have structural NaN")
print(f"  from pre-1984 records — left as NaN (historical data not collected).")

# Final NaN count
print("\nFinal missing values after interpolation:")
print(df_s2.isnull().sum())
print(f"Total NaNs (final): {df_s2.isnull().sum().sum()}")

print(f"\nFinal shape: {df_s2.shape}")
print("\nFirst 5 rows:")
print(df_s2.head())


Data Cleaning for Source 2: Singapore Air Temperature

Initial missing values (NaN):
month                                                                0
air_temperature_means_daily_maximum_degree_celsius                   0
air_temperature_means_daily_minimum_degree_celsius                   0
air_temperature_absolute_extremes_maximum_degree_celsius             0
air_temperature_absolute_extremes_minimum_degree_celsius             0
total_rainfall_millimetre                                            0
highest_daily_rainfall_total_millimetre                              0
number_of_rainy_days_number                                          0
bright_sunshine_daily_mean_hour                                      0
minimum_relative_humidity_per_cent                                 108
24_hours_mean_relative_humidity_per_cent                             0
year                                                                 0
month_num                                                      

##### 2.3.2.5 Remarks for Source 2

*   Not much insights from Temperature Change due to its high Stability
*   Monthly Data Masks Extreme Events (For example, one week of extremely heavy rain is being averaged out, which might lead to overfitting)
*   Able to be crossed reference against Source 1 for more reliability.
*   **Take note that minimum relative humidity has 108 NaN values for the earlier records (in 1975s onwards).**

##### 2.3.2.6 Conclusion

We will use DS2!

This dataset is high-quality, well-suited for our research question, and provides an excellent foundation. The strong rainfall variation and perfect monthly alignment with tourism data make it ideal for detecting weather-tourism relationships.
However, we will need to supplement it with air quality data and control variables to discover more robust correlations between climates and tourism in Singapore.




#### 2.3.3 Dataset 3 (DS3): Singapore Tourist Arrival

Description: The dataset provides a time-series record of visitor numbers to Singapore, broken down by the visitors' country or region of residence. The rows represent the hierarchy of visitor origins (e.g., "Southeast Asia" > "Indonesia"). It captures the volume of tourism, a critical economic indicator. The values are counts of persons.

##### 2.3.3.1  Data Profiling for Source 3

In [23]:
print("Data Profiling for Source 3: Tourist Arrivals")
print("="*60)

file_path3 = 'https://docs.google.com/spreadsheets/d/e/2PACX-1vRNnq5t44f8KjxdzKEedZtn2ehSbvX6gyzRpBT_jn_NArlr7xnGXUpNrvq72LiERA/pub?output=csv'
df_s3_raw = pd.read_csv(file_path3)

df = df_s3_raw

print("\n1. STRUCTURE INSPECTION")
print("-" * 60)
print(f"Source 3 has {df.shape[0]} rows and {df.shape[1]} columns")

print("\n2. DATA TYPES & MISSING VALUES")
print("-" * 60)
print("\nColumn Data Types:")
print(df.dtypes)
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_summary = pd.DataFrame({'Missing Count': missing, 'Percentage': missing_pct})
print("\nMissing Values:")
print(missing_summary[missing_summary['Missing Count'] > 0])
print(f"\nTotal data points   : {df.size}")
print(f"Missing data points : {df.isnull().sum().sum()}")
print(f"Data completeness   : {(1 - df.isnull().sum().sum()/df.size)*100:.2f}%")

print("\n3. STATISTICAL PROFILING (Numeric Columns)")
print("-" * 60)
# Values are comma-formatted strings in wide format, check after structuring
print("Note: Arrival values are comma-formatted strings in wide format.")
print("Full statistical profiling will be available after Stage 2 structuring.")
print(f"\nNumber of month columns : {df.shape[1] - 1}")
print(f"Data Series rows        : {df.shape[0]}")
print("\nData Series found:")
print(df.iloc[:, 0].tolist())

print("\n4. CATEGORICAL COLUMNS ANALYSIS")
print("-" * 60)
print(f"First column (Data Series): {df.iloc[:, 0].nunique()} unique series")
print(df.iloc[:, 0].value_counts())

print("\n5. DATE/DATETIME ANALYSIS")
print("-" * 60)
month_cols = df.columns[1:].tolist()
print(f"Month columns span: {month_cols[-1]} to {month_cols[0]}")
print(f"Total months available: {len(month_cols)}")

print("\n6. DATA QUALITY CHECKS")
print("-" * 60)
print(f"Duplicate rows: {df.duplicated().sum()}")
print("\nSample of raw values (first 5 months):")
print(df.iloc[0, 1:6].tolist())

print("\n7. CORRELATION ANALYSIS (Top Numeric Columns)")
print("-" * 60)
print("Not applicable in wide format — available after Stage 2 structuring.")

print("\n8. UNIQUE IDENTIFIERS CHECK")
print("-" * 60)
unique_ratio = df.iloc[:, 0].nunique() / len(df)
if unique_ratio == 1.0:
    print(f"'{df.columns[0]}' - Potential unique identifier (100% unique)")
else:
    print(f"'{df.columns[0]}' - Uniqueness: {unique_ratio*100:.1f}%")

print("\n9. DATA QUALITY DISCUSSION (Source 3: Tourist Arrivals)")
print("-" * 60)

duplicates = df.duplicated().sum()
print("\nAccuracy & Reputation:")
if duplicates == 0:
    print("  (+) No duplicate rows detected, indicating consistent data recording.")
else:
    print(f"  (-) {duplicates} duplicate rows found, which may distort analysis.")

print("  (+) Dataset is sourced from an official tourism authority, increasing reliability of reported figures.")

month_cols = df.columns[1:]
num_months = len(month_cols)

print("\nTimeliness:")
print(f"  (+) The dataset contains {num_months} monthly observations.")
print("      → High temporal granularity enables seasonal and trend analysis.")

# --- Consistency / Structure (Main Issue) ---
print("\nConsistency:")
print("  (-) Data is stored in wide format with months as columns and requires restructuring before analysis.")
print("  (-) Numeric values are stored as comma-formatted strings (e.g. '1,234') and we need to convert it to numeric format before statistical analysis.")


Data Profiling for Source 3: Tourist Arrivals

1. STRUCTURE INSPECTION
------------------------------------------------------------
Source 3 has 62 rows and 218 columns

2. DATA TYPES & MISSING VALUES
------------------------------------------------------------

Column Data Types:
Data Series    str
2026 Jan       str
2025 Dec       str
2025 Nov       str
2025 Oct       str
              ... 
2008 May       str
2008 Apr       str
2008 Mar       str
2008 Feb       str
2008 Jan       str
Length: 218, dtype: object

Missing Values:
Empty DataFrame
Columns: [Missing Count, Percentage]
Index: []

Total data points   : 13516
Missing data points : 0
Data completeness   : 100.00%

3. STATISTICAL PROFILING (Numeric Columns)
------------------------------------------------------------
Note: Arrival values are comma-formatted strings in wide format.
Full statistical profiling will be available after Stage 2 structuring.

Number of month columns : 217
Data Series rows        : 62

Data Series foun

##### 2.3.3.2  Data Structuring for Source 3

In [24]:
print("Data Structuring for Source 3: Tourist Arrivals")
print("="*60)

# Preview shape
print(f"\nRaw shape: {df_s3_raw.shape}")

df_s3_total = df_s3_raw[
    df_s3_raw['Data Series'].str.strip() == 'Total International Visitor Arrivals By Place Of Residence'
].copy()

# Transpose columns into rows
df_s3_total = df_s3_total.set_index('Data Series').T.reset_index()
df_s3_total.columns = ['month_str', 'tourist_arrivals']

# Remove commas and convert to numeric
df_s3_total['tourist_arrivals'] = (
    df_s3_total['tourist_arrivals']
    .astype(str)
    .str.replace(',', '', regex=False)
)
df_s3_total['tourist_arrivals'] = pd.to_numeric(df_s3_total['tourist_arrivals'], errors='coerce')

# Parse month string (e.g. "2024 Jan") to period
df_s3_total['month'] = pd.to_datetime(
    df_s3_total['month_str'].str.strip(), format='%Y %b', errors='coerce'
).dt.to_period('M')

invalid_months = df_s3_total['month'].isna().sum()
print(f"Invalid month entries removed : {invalid_months}")
df_s3_total = df_s3_total[df_s3_total['month'].notna()].copy()
df_s3_total = df_s3_total.drop(columns=['month_str'])
df_s3_total = df_s3_total[['month', 'tourist_arrivals']]

print(f"Columns   : {df_s3_total.columns.tolist()}")
print(f"Shape     : {df_s3_total.shape}")
print(f"Date range: {df_s3_total['month'].min()} to {df_s3_total['month'].max()}")
print("\nFirst 5 rows:")
print(df_s3_total.head())

df_s3_all = df_s3_raw.copy()

df_s3_all['Data Series'] = df_s3_all['Data Series'].str.strip()

df_s3_all = df_s3_all.set_index('Data Series').T.reset_index()
df_s3_all.rename(columns={'index': 'month_str'}, inplace=True)

df_s3_all['month'] = pd.to_datetime(
    df_s3_all['month_str'].str.strip(), format='%Y %b', errors='coerce'
).dt.to_period('M')
df_s3_all = df_s3_all[df_s3_all['month'].notna()].drop(columns=['month_str'])

region_cols = [c for c in df_s3_all.columns if c != 'month']
for col in region_cols:
    df_s3_all[col] = pd.to_numeric(
        df_s3_all[col].astype(str).str.replace(',', '', regex=False),
        errors='coerce'
    )

df_s3_all = df_s3_all[['month'] + region_cols]

print(f"\n[Option 2] All regions retained, for regional breakdown subsequently when needed")
print(f"Columns   : {df_s3_all.columns.tolist()}")
print(f"Shape     : {df_s3_all.shape}")
print(f"Date range: {df_s3_all['month'].min()} to {df_s3_all['month'].max()}")
print("\nFirst 5 rows:")
print(df_s3_all.head())

df_s3 = df_s3_total.copy()
print(f"\nActive dataframe set to: df_s3_total (Total arrivals only)")

Data Structuring for Source 3: Tourist Arrivals

Raw shape: (62, 218)
Invalid month entries removed : 0
Columns   : ['month', 'tourist_arrivals']
Shape     : (217, 2)
Date range: 2008-01 to 2026-01

First 5 rows:
     month  tourist_arrivals
0  2026-01           1501420
1  2025-12           1363999
2  2025-11           1293401
3  2025-10           1379077
4  2025-09           1254346

[Option 2] All regions retained, for regional breakdown subsequently when needed
Columns   : ['month', 'Total International Visitor Arrivals By Place Of Residence', 'Southeast Asia', 'Brunei Darussalam', 'Indonesia', 'Malaysia', 'Myanmar', 'Philippines', 'Thailand', 'Vietnam', 'Other Markets In Southeast Asia', 'Greater China', 'China', 'Hong Kong SAR', 'Taiwan', 'Other Markets In Greater China', 'North Asia', 'Japan', 'South Korea', 'Other Markets In North Asia', 'South Asia', 'Bangladesh', 'India', 'Pakistan', 'Sri Lanka', 'Other Markets In South Asia', 'West Asia', 'Iran', 'Israel', 'Kuwait', 'Saudi Ar

##### 2.3.3.3  Data Enriching for Source 3

In [25]:
print("Data Enriching for Source 3: Tourist Arrivals")
print("="*60)

df_s3 = df_s3.sort_values('month').reset_index(drop=True)

# Extract year and month number
df_s3['year']      = df_s3['month'].dt.year
df_s3['month_num'] = df_s3['month'].dt.month

# Singapore meteorological season
df_s3['season'] = df_s3['month_num'].apply(get_season)

# Year-over-year growth rate (compared to same month prior year)
df_s3['arrivals_yoy_growth_pct'] = (
    df_s3['tourist_arrivals'].pct_change(12) * 100
).round(2)

# 3-month rolling average to reduce the noise in data
df_s3['arrivals_3m_rolling_avg'] = (
    df_s3['tourist_arrivals'].rolling(window=3, min_periods=1).mean().round(0)
)

df_s3['covid_period'] = df_s3['month'].apply(
    lambda m: 1 if pd.Period('2020-01', 'M') <= m <= pd.Period('2021-12', 'M') else 0
)

print(f"New columns added : year, month_num, season, arrivals_yoy_growth_pct, arrivals_3m_rolling_avg, covid_period")
print(f"Shape             : {df_s3.shape}")
print("\nFirst 5 rows:")
print(df_s3.head())

Data Enriching for Source 3: Tourist Arrivals
New columns added : year, month_num, season, arrivals_yoy_growth_pct, arrivals_3m_rolling_avg, covid_period
Shape             : (217, 8)

First 5 rows:
     month  tourist_arrivals  year  month_num             season  \
0  2008-01            885290  2008          1  Northeast Monsoon   
1  2008-02            813028  2008          2  Northeast Monsoon   
2  2008-03            910635  2008          3    Inter-monsoon 1   
3  2008-04            829438  2008          4    Inter-monsoon 1   
4  2008-05            833604  2008          5  Southwest Monsoon   

   arrivals_yoy_growth_pct  arrivals_3m_rolling_avg  covid_period  
0                      NaN                 885290.0             0  
1                      NaN                 849159.0             0  
2                      NaN                 869651.0             0  
3                      NaN                 851034.0             0  
4                      NaN                 857892.0  

##### 2.3.3.4  Data Cleaning for Source 3

In [26]:
print("Data Cleaning for Source 3: Tourist Arrivals")
print("="*60)

# Remove duplicate months
before = len(df_s3)
df_s3 = df_s3.drop_duplicates(subset='month').sort_values('month').reset_index(drop=True)
print(f"\nDuplicate months removed : {before - len(df_s3)}")

invalid_arrivals = (df_s3['tourist_arrivals'] < 0).sum()
df_s3.loc[df_s3['tourist_arrivals'] < 0, 'tourist_arrivals'] = np.nan
print(f"Negative arrival values replaced with NaN : {invalid_arrivals}")

# Interpolate missing values, less covid
print("\nInterpolating missing values...")
df_s3[['tourist_arrivals', 'arrivals_3m_rolling_avg']] = (
    df_s3[['tourist_arrivals', 'arrivals_3m_rolling_avg']]
    .interpolate(method='linear', limit_direction='both')
)

print(f"Remaining nulls : {df_s3.isnull().sum().sum()}")
print(f"Final shape     : {df_s3.shape}")
print("\nFirst 5 rows:")
print(df_s3.head())

Data Cleaning for Source 3: Tourist Arrivals

Duplicate months removed : 0
Negative arrival values replaced with NaN : 0

Interpolating missing values...
Remaining nulls : 12
Final shape     : (217, 8)

First 5 rows:
     month  tourist_arrivals  year  month_num             season  \
0  2008-01          885290.0  2008          1  Northeast Monsoon   
1  2008-02          813028.0  2008          2  Northeast Monsoon   
2  2008-03          910635.0  2008          3    Inter-monsoon 1   
3  2008-04          829438.0  2008          4    Inter-monsoon 1   
4  2008-05          833604.0  2008          5  Southwest Monsoon   

   arrivals_yoy_growth_pct  arrivals_3m_rolling_avg  covid_period  
0                      NaN                 885290.0             0  
1                      NaN                 849159.0             0  
2                      NaN                 869651.0             0  
3                      NaN                 851034.0             0  
4                      NaN        




**Data Quality**
1. Positive Criterion: Internal Consistency

The dataset exhibits high internal consistency. As demonstrated in the profiling step, the hierarchical aggregations are accurate. The sum of visitor counts from individual countries (e.g., Indonesia, Malaysia, etc.) matches the reported regional total (Southeast Asia). This reliability allows analysts to confidently use either the granular data for detail or the aggregate data for high-level overviews without fear of conflicting numbers.

2. Problematic Criterion: Structure

The dataset scores poorly on processing since it is designed for humans to read rather than algorithmic processing.

The Place of Residence column mixes different levels of hierarchy (Total, Region, Country) in a single column, distinguished only by leading whitespace (indentation). Thus, we cannot simply group by this column. Without parsing the indentation to create separate "Region" and "Country" columns, a naive summation would triple-count visitors (Country + Region + Total).

**Conclusion**

This dataset is highly suitable for analysing international visitor trends to Singapore, given that we already have conducted specific preprocessing steps are taken. The data is timely and consistent. However, we must be careful to filter out metadata footers and handle the hierarchical structure to avoid double-counting. Additionally, any time-series forecasting models must account for the structural break caused by COVID-19 in 2020-2021, where values dropped to near zero, as these outliers could skew standard trend predictions.